> Adapted from *Dive into Deep Learning*, section [Custom Layers](https://d2l.ai/chapter_builders-guide/custom-layer.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 6.4: Lazy Initialization](06.04_lazy-init.ipynb) | Next: [Section 6.6: File I/O](06.06_read-write.ipynb)

# Custom Layers

One factor behind deep learning's success
is the availability of a wide range of layers
that can be composed in creative ways
to design architectures suitable
for a wide variety of tasks.
For instance, researchers have invented layers
specifically for handling images, text,
looping over sequential data,
and
performing dynamic programming.
Sooner or later, you will need
a layer that does not exist yet in the deep learning framework.
In these cases, you must build a custom layer.
In this section, we show you how.

In [ ]:
import numpy as np
from divelet_ml import nn

import divelet_ml as d2l

## Layers without Parameters

To start, we construct a custom layer
that does not have any parameters of its own.
This should look familiar if you recall our
introduction to modules in [Section 6.1](https://d2l.ai/chapter_builders-guide/model-construction.html#sec-model-construction).
The following `CenteredLayer` class simply
subtracts the mean from its input.
To build it, we simply need to inherit
from the base layer class and implement the forward propagation function.

In [ ]:
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

Let's verify that our layer works as intended by feeding some data through it.

In [ ]:
layer = CenteredLayer()
layer(np.array([1.0, 2, 3, 4, 5]))

We can now [**incorporate our layer as a component
in constructing more complex models.**]

In [ ]:
net = nn.Sequential(nn.LazyLinear(128), CenteredLayer())

As an extra sanity check, we can send random data
through the network and check that the mean is in fact 0.
Because we are dealing with floating point numbers,
we may still see a very small nonzero number
due to quantization.

In [ ]:
Y = net(np.random.rand(4, 8))
Y.mean()

## Layers with Parameters

Now that we know how to define simple layers,
let's move on to defining layers with parameters
that can be adjusted through training.
In `divelet_ml`, a floating-point array held as an attribute of a module is a parameter, which
provides some basic housekeeping functionality.
In particular, `parameters()`, `state_dict()` and `d2l.param_grads` find it, which govern access, initialization,
sharing, saving, and loading model parameters.
This way, among other benefits, we will not need to write
custom serialization routines for every custom layer.

Now let's implement our own version of the  fully connected layer.
Recall that this layer requires two parameters,
one to represent the weight and the other for the bias.
In this implementation, we bake in the ReLU activation as a default.
This layer requires two input arguments: `in_units` and `units`, which
denote the number of inputs and outputs, respectively.

In [ ]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = np.random.randn(in_units, units)
        self.bias = np.random.randn(units,)
        
    def forward(self, X):
        linear = np.matmul(X, self.weight) + self.bias
        return nn.relu(linear)

Next, we instantiate the `MyLinear` class
and access its model parameters.

In [ ]:
linear = MyLinear(5, 3)
linear.weight

We can directly carry out forward propagation calculations using custom layers.

In [ ]:
linear(np.random.rand(2, 5))

We can also construct models using custom layers.
Once we have that we can use it just like the built-in fully connected layer.

In [ ]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(np.random.rand(2, 64))

## Summary

We can design custom layers via the basic layer class. This allows us to define flexible new layers that behave differently from any existing layers in the library.
Once defined, custom layers can be invoked in arbitrary contexts and architectures.
Layers can have local parameters, which can be created through built-in functions.


## Exercises

1. Design a layer that takes an input and computes a tensor reduction,
   i.e., it returns $y_k = \sum_{i, j} W_{ijk} x_i x_j$.
1. Design a layer that returns the leading half of the Fourier coefficients of the data.



Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/59)

Previous: [Section 6.4: Lazy Initialization](06.04_lazy-init.ipynb) | Next: [Section 6.6: File I/O](06.06_read-write.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[Custom Layers](https://d2l.ai/chapter_builders-guide/custom-layer.html)
(source: [`chapter_builders-guide/custom-layer.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_builders-guide/custom-layer.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python). The word *tensor* is kept and means a NumPy `ndarray`.
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`.
- `nn.Parameter(torch.randn(...))` is `np.random.randn(...)`: an array held as an attribute is a parameter, so `.data` is dropped; `torch.matmul` is `np.matmul` and `F.relu` is `nn.relu`; `d2l.tensor` and `d2l.rand` are `np.array` and `np.random.rand`.
- Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.